### Imports

In [4]:
from fundus_dataset import AugmentPair, FundusVesselDataset, CenterCropPair
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from safetensors.torch import save_file
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### Testing the Dataset

In [15]:
img_dir = "fundus/train/Original/"
mask_dir = "fundus/train/Ground truth"

train_full = FundusVesselDataset(
    img_dir=img_dir,
    mask_dir=mask_dir,
    transform=AugmentPair(crop_size=(512, 512)),
)

val_full = FundusVesselDataset(
    img_dir=img_dir,
    mask_dir=mask_dir,
    transform=None,
)

num_samples = len(train_full)
generator = torch.Generator().manual_seed(42)

# Shuffle the indices
indices = torch.randperm(num_samples, generator=generator).tolist()

train_size = int(0.8 * num_samples)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

images, masks = next(iter(train_loader))
val_images, val_masks = next(iter(val_loader))

print("Train images:", images.shape, images.dtype, images.min().item(), images.max().item())
print("Train masks: ", masks.shape, masks.dtype, torch.unique(masks))

print("Val images:", val_images.shape, val_images.dtype, val_images.min().item(), val_images.max().item())
print("Val masks: ", val_masks.shape, val_masks.dtype, torch.unique(val_masks))

Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Train size: 480
Val size: 120


Train images: torch.Size([4, 3, 512, 512]) torch.float32 0.0 1.0
Train masks:  torch.Size([4, 1, 512, 512]) torch.float32 tensor([0., 1.])
Val images: torch.Size([1, 3, 2048, 2048]) torch.float32 0.0 1.0
Val masks:  torch.Size([1, 1, 2048, 2048]) torch.float32 tensor([0., 1.])


### BCEDice Loss Functions

In [3]:
class BCEDiceLoss(nn.Module):
    def __init__(self, w_bce=0.5, w_dice=0.5):
        super().__init__()
        self.w_bce = w_bce
        self.w_dice = w_dice

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss(sigmoid=True)

    def forward(self, logits, masks):
        bce = self.bce(logits, masks)
        dice = self.dice(logits, masks)

        return self.w_bce * bce + self.w_dice * dice

def hard_dice_score(logits, masks, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    # flatten each image separately: [B, 1, H, W] -> [B, pixels]
    preds = preds.flatten(start_dim=1)
    masks = masks.flatten(start_dim=1)

    intersection = (preds * masks).sum(dim=1)
    denominator = preds.sum(dim=1) + masks.sum(dim=1)

    dice = (2 * intersection + eps) / (denominator + eps)

    return dice.mean()

### U-Net Factory

In [6]:
def make_unet(channels=(16, 32, 64, 128, 256)):
    model = UNet(
        spatial_dims=2,
        in_channels=3,
        out_channels=1,
        channels=channels,
        strides=(2, 2, 2, 2),
        num_res_units=2,
    )
    return model.to(device)

### Training Loop Function

In [5]:
def validate_full_image(model, loader, loss_fn, threshold=0.5):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = loss_fn(logits, masks)
            dice = hard_dice_score(logits, masks, threshold=threshold)

            val_loss += loss.item() * images.size(0)
            val_dice += dice.item() * images.size(0)

    val_loss /= len(loader.dataset)
    val_dice /= len(loader.dataset)
    return val_loss, val_dice


def train_model(model, train_loader, val_loader, loss_fn, optimizer, epochs, save_path=None, save_name="baseline.safetensors"):
    history = []
    best_val_dice = 0.0
    best_epoch = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_loss_steps = []

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = loss_fn(logits, masks)

            loss.backward()
            optimizer.step()

            step_loss = loss.item()
            train_loss += step_loss * images.size(0)
            train_loss_steps.append(step_loss)

        train_loss /= len(train_loader.dataset)

        # Default validation protocol: full-image metrics on val_loader.
        val_loss, val_dice = validate_full_image(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            threshold=0.5,
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_loss_steps": train_loss_steps,
            "val_loss": val_loss,
            "val_dice": val_dice,
        })

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_epoch = epoch + 1

            if save_path is not None:
                save_dir = Path(save_path)
                save_dir.mkdir(parents=True, exist_ok=True)
                st_path = save_dir / save_name
                save_file(model.state_dict(), st_path)

        print(
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"Best: {best_val_dice:.4f} @ {best_epoch}"
        )

    return history

### DANGEROUS (RESET EXPERIMENT)

In [6]:
model = make_unet(channels=(32, 64, 128, 256, 512))
loss_fn = BCEDiceLoss(w_bce=0.5, w_dice=0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100
SAVE_PATH = "/workspace/models_baseline/"
SAVE_NAME = "baseline.safetensors"

### START OR CONTINUE EXPERIMENT

In [7]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    save_path=SAVE_PATH,
    save_name=SAVE_NAME,
)

Epoch 001/100 | Train loss: 0.6968 | Val loss: 0.6780 | Val Dice: 0.6360 | Best: 0.6360 @ 1
Epoch 002/100 | Train loss: 0.6364 | Val loss: 0.6266 | Val Dice: 0.6560 | Best: 0.6560 @ 2
Epoch 003/100 | Train loss: 0.5776 | Val loss: 0.5630 | Val Dice: 0.7270 | Best: 0.7270 @ 3
Epoch 004/100 | Train loss: 0.5156 | Val loss: 0.4924 | Val Dice: 0.7755 | Best: 0.7755 @ 4
Epoch 005/100 | Train loss: 0.4407 | Val loss: 0.4312 | Val Dice: 0.7700 | Best: 0.7755 @ 4
Epoch 006/100 | Train loss: 0.3859 | Val loss: 0.3670 | Val Dice: 0.7566 | Best: 0.7755 @ 4
Epoch 007/100 | Train loss: 0.3265 | Val loss: 0.2956 | Val Dice: 0.7994 | Best: 0.7994 @ 7
Epoch 008/100 | Train loss: 0.2615 | Val loss: 0.2196 | Val Dice: 0.8378 | Best: 0.8378 @ 8
Epoch 009/100 | Train loss: 0.2280 | Val loss: 0.1853 | Val Dice: 0.8360 | Best: 0.8378 @ 8
Epoch 010/100 | Train loss: 0.1898 | Val loss: 0.1554 | Val Dice: 0.8492 | Best: 0.8492 @ 10
Epoch 011/100 | Train loss: 0.1724 | Val loss: 0.1372 | Val Dice: 0.8578 | Best

### Save your Settings

In [9]:
import pandas as pd

history_df = pd.DataFrame(history)
history_df.to_csv("/workspace/models_baseline/baseline_history.csv", index=False)